# NOTEBOOK4 semaine 1er décembre

Questions et perspectives de travail:
- faire un état des lieux de l'efficacité de SHAP, LIME, Permutation, Ridge, Lasso, ElasticNet pour les variables confondantes et les features inutiles
- étendre au cas de chaines de plus de trois variables
- comprendre pourquoi certaines méthodes sont plus fonctionnelles que d'autres
- tenter le cas du causal shap

## STEP 1: état des lieux

### DATA SET avec variable confondante (chaine de 3)

In [ ]:
#importer

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import shap
import eli5
from eli5.sklearn import PermutationImportance
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import lime
import lime.lime_tabular


In [ ]:
#generate data

d = {}
d["f1"] = np.random.randn(800)
d["f2"] = d["f1"]*10 + np.random.randn(800)*0.1
for i in range(3, 6):
    d[f"f{i}"] = np.random.randn(800)
d["f6"] = 0.6*d["f1"] + 6*d["f2"] + 0.2*d["f3"] - 0.1*d["f4"] + 3

df = pd.DataFrame(d)

X = df.drop("f6", axis=1)
y = df["f6"]

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
model = RandomForestRegressor(random_state=55)
model.fit(train_X, train_y)

In [ ]:
#shap

shap.initjs()
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(val_X)
shap.summary_plot(sv, val_X)
shap.summary_plot(sv, val_X, plot_type="bar")

SHAP n'identifie pas la variable confondante

In [ ]:
#LIME

preds=model.predict(val_X)

explainer = lime.lime_tabular.LimeTabularExplainer(train_X.values, feature_names=train_X.columns.values.tolist(),  verbose=True, mode='regression')

exp = explainer.explain_instance(val_X.values[0], model.predict, num_features=6)

html = exp.as_html(show_table=True)
with open("lime_explanation.html", "w") as f:
    f.write(html)

LIME non plus

In [ ]:
#permutation

perm = PermutationImportance(model, random_state=1).fit(val_X, val_y)
eli5.show_weights(perm, feature_names=val_X.columns.tolist())


Permutation non plus

In [ ]:
#LASSO, ridge, elasticnet

lasso = Pipeline([("scaler", StandardScaler()), ("m", Lasso(alpha=0.1))])
ridge = Pipeline([("scaler", StandardScaler()), ("m", Ridge(alpha=1.0))])
enet = Pipeline([("scaler", StandardScaler()), ("m", ElasticNet(alpha=0.1, l1_ratio=0.5))])

lasso.fit(train_X, train_y)
ridge.fit(train_X, train_y)
enet.fit(train_X, train_y)

lasso_coefs = dict(zip(train_X.columns, lasso.named_steps["m"].coef_))
ridge_coefs = dict(zip(train_X.columns, ridge.named_steps["m"].coef_))
enet_coefs = dict(zip(train_X.columns, enet.named_steps["m"].coef_))

lasso_coefs, ridge_coefs, enet_coefs


lasso perfectly id coeff of confounding variable, ridge and elastic net don't

### DATA AVEC VARIABLE CONFONDANTE, CHAINE DE 4

In [ ]:
#generate data

d = {}
d["f1"] = np.random.randn(800)
d["f2"] = d["f1"]*10 + np.random.randn(800)*0.1
d["f3"] = d["f2"]*5 + np.random.randn(800)*0.2
for i in range(4, 6):
    d[f"f{i}"] = np.random.randn(800)
d["f6"] = 0.6*d["f1"] + 0.3*d["f2"] + 6*d["f3"] - 0.1*d["f4"] + 3

df = pd.DataFrame(d)

X = df.drop("f6", axis=1)
y = df["f6"]

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
model = RandomForestRegressor(random_state=55)
model.fit(train_X, train_y)

In [ ]:
#shap

shap.initjs()
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(val_X)
shap.summary_plot(sv, val_X)
shap.summary_plot(sv, val_X, plot_type="bar")

toujours pas, on tente avec LASSO

In [ ]:
#lasso

lasso = Pipeline([("scaler", StandardScaler()), ("m", Lasso(alpha=0.1))])
lasso.fit(train_X, train_y)
lasso_coefs = dict(zip(train_X.columns, lasso.named_steps["m"].coef_))

lasso_coefs


0.6+ 0.3x10 + 6x5x10= 303.6, pas loin de 274 mais moins bon, compte double f2 (28 proche de 6x5)
Rententons l'expérience

In [ ]:
d = {}
d["f1"] = np.random.randn(800)
d["f2"] = d["f1"]*10 + np.random.randn(800)*0.1
d["f3"] = d["f2"]*5 + np.random.randn(800)*0.2
for i in range(4, 6):
    d[f"f{i}"] = np.random.randn(800)
d["f6"] = 0.2*d["f1"] + 0.1*d["f2"] + 10*d["f3"] - 0.1*d["f4"] + 3

df = pd.DataFrame(d)

X = df.drop("f6", axis=1)
y = df["f6"]

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

lasso = Pipeline([("scaler", StandardScaler()), ("m", Lasso(alpha=0.1))])
lasso.fit(train_X, train_y)
lasso_coefs = dict(zip(train_X.columns, lasso.named_steps["m"].coef_))

lasso_coefs

moins bon aussi. tentons chaine de 4

In [ ]:
d = {}
d["f1"] = np.random.randn(800)
d["f2"] = d["f1"]*10 + np.random.randn(800)*0.1
d["f3"] = d["f2"]*5 + np.random.randn(800)*0.2
d["f4"] = d["f3"]*8 + np.random.randn(800)*0.2
for i in range(5, 6):
    d[f"f{i}"] = np.random.randn(800)
d["f6"] = 0.2*d["f1"] + 0.1*d["f2"] + 10*d["f3"] + 10*d["f4"] + 3

df = pd.DataFrame(d)

X = df.drop("f6", axis=1)
y = df["f6"]

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

lasso = Pipeline([("scaler", StandardScaler()), ("m", Lasso(alpha=0.1))])
lasso.fit(train_X, train_y)
lasso_coefs = dict(zip(train_X.columns, lasso.named_steps["m"].coef_))

lasso_coefs

ok intéressant. identifie bien que f3->f4 (8x10), que f2->f3, mais sous évalue f1 (4028 au lieu de 4500)